# Phase 2 — Variants & Scores Report

**Covers:**
1. Training data (80K ClinVar) — score availability & distributions
2. Test data (~1M CRISPR variants) — score availability
3. 22 Phase 1 Galaxy variants — complete score table + heatmap

**Scores:** CADD Phred, GERP RS, SIFT, PolyPhen2, SpliceAI

All scores available for future researchers to use independently.
ML model was trained on **cadd_phred only** (single feature, AUC~0.886).

In [ ]:
!pip install pandas numpy matplotlib seaborn -q
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import warnings, os
warnings.filterwarnings('ignore')
os.makedirs('figures_scores', exist_ok=True)
print('✅ Libraries ready')

In [ ]:
from google.colab import drive
import pandas as pd, os

drive.mount('/content/drive')

TRAIN_PATH = '/content/drive/My Drive/clinvar_training_data_v2.csv'
TEST_PATH  = '/content/drive/My Drive/all_variants_enriched_step3.csv'

print('Loading training data...')
train = pd.read_csv(TRAIN_PATH)
print(f'✅ Training : {len(train):,}  columns: {list(train.columns)}')

print('Loading test data...')
test = pd.read_csv(TEST_PATH, low_memory=False)
print(f'✅ Test data: {len(test):,}  columns: {list(test.columns)}')

## Section 1 — Training Data Score Availability

In [ ]:
SCORE_COLS   = ['cadd_phred','gerp_rs','sift_score','polyphen_score','spliceai_score']
SCORE_LABELS = ['CADD Phred','GERP RS','SIFT','PolyPhen2','SpliceAI']

print('TRAINING DATA — SCORE AVAILABILITY')
print('='*70)
print(f'Total: {len(train):,}  |  Pathogenic: {(train["label"]==1).sum():,}  |  Benign: {(train["label"]==0).sum():,}')
print()
print(f'{"Score":<22} {"Non-null":>10} {"% avail":>9} {"P mean":>9} {"B mean":>9} {"Used for ML":>13}')
print('-'*75)
for col in SCORE_COLS:
    if col not in train.columns:
        print(f'  {col:<20}  NOT IN DATA')
        continue
    nn   = train[col].notna().sum()
    pct  = nn/len(train)*100
    p    = train[train['label']==1][col].mean()
    b    = train[train['label']==0][col].mean()
    ml   = '✅ YES' if col=='cadd_phred' else '— No'
    print(f'  {col:<20} {nn:>10,} {pct:>8.1f}% {p:>9.3f} {b:>9.3f} {ml:>13}')
print()
print('Note: SIFT/PolyPhen apply to missense (~80%). SpliceAI applies to splice (~20%).')
print('      CADD & GERP cover ALL variant types.')

In [ ]:
# Score distribution plots — training data
import matplotlib.pyplot as plt

avail = [c for c in SCORE_COLS if c in train.columns and train[c].notna().sum()>100]
fig, axes = plt.subplots(1, len(avail), figsize=(4*len(avail), 5))
if len(avail)==1: axes=[axes]
fig.suptitle('ClinVar Training Data — Score Distributions (Pathogenic vs Benign)', fontsize=12, fontweight='bold')

for ax, col, lbl in zip(axes, avail, [l for c,l in zip(SCORE_COLS,SCORE_LABELS) if c in avail]):
    ax.hist(train[train['label']==1][col].dropna(), bins=40, alpha=0.65, color='#C0392B', label='Pathogenic', density=True)
    ax.hist(train[train['label']==0][col].dropna(), bins=40, alpha=0.65, color='#27AE60', label='Benign',     density=True)
    ax.set_title(lbl, fontweight='bold', fontsize=10)
    ax.set_xlabel(col, fontsize=9)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('figures_scores/training_score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved')

## Section 2 — Test Data Score Availability

In [ ]:
print('TEST DATA (~1M CRISPR VARIANTS) — SCORE AVAILABILITY')
print('='*70)
print(f'Total: {len(test):,}')
print()
print(f'{"Score":<22} {"Non-null":>10} {"% avail":>9} {"Mean":>8} {"Std":>8} {"Min":>7} {"Max":>7}')
print('-'*75)
for col in SCORE_COLS:
    if col not in test.columns:
        print(f'  {col:<20}  NOT IN TEST DATA')
        continue
    v = test[col].dropna()
    print(f'  {col:<20} {len(v):>10,} {len(v)/len(test)*100:>8.1f}% {v.mean():>8.3f} {v.std():>8.3f} {v.min():>7.2f} {v.max():>7.2f}')
print()
print('Impact breakdown:')
print(test['impact_severity'].value_counts().to_string())

In [ ]:
# Test data score distributions by impact
import matplotlib.pyplot as plt

avail_t = [c for c in SCORE_COLS if c in test.columns and test[c].notna().sum()>100]
fig, axes = plt.subplots(1, len(avail_t), figsize=(4*len(avail_t), 5))
if len(avail_t)==1: axes=[axes]
fig.suptitle('Test Data (~1M CRISPR Variants) — Score Distributions by Impact', fontsize=12, fontweight='bold')
cols_imp = {'HIGH':'#C0392B','MED':'#E67E22','LOW':'#27AE60'}
for ax, col, lbl in zip(axes, avail_t, [l for c,l in zip(SCORE_COLS,SCORE_LABELS) if c in avail_t]):
    for imp, color in cols_imp.items():
        vals = test[test['impact_severity']==imp][col].dropna()
        if len(vals)>0:
            ax.hist(vals.sample(min(len(vals),20000),random_state=42), bins=40,
                    alpha=0.55, color=color, label=f'{imp}(n={len(vals):,})', density=True)
    ax.set_title(lbl, fontweight='bold', fontsize=10)
    ax.set_xlabel(col, fontsize=9)
    ax.set_ylabel('Density')
    ax.legend(fontsize=7)
    ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('figures_scores/test_score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved')

## Section 3 — 22 Phase 1 Variants: Complete Score Table + Heatmap

In [ ]:
import pandas as pd

variants_22 = pd.DataFrame([
    {'#':1,  'gene':'HP1BP3', 'chrom':'chr1',  'pos':21072082,  'ref':'A',  'alt':'G',  'variant_type':'missense_variant',       'impact':'MODERATE','cadd_phred':23.5,  'gerp_rs':5.87,  'sift_score':0.15, 'polyphen_score':0.91, 'spliceai_score':0.01},
    {'#':2,  'gene':'AGAP4',  'chrom':'chr10', 'pos':46342157,  'ref':'G',  'alt':'T',  'variant_type':'missense_variant',       'impact':'MODERATE','cadd_phred':4.63,  'gerp_rs':1.34,  'sift_score':0.03, 'polyphen_score':0.02, 'spliceai_score':0.01},
    {'#':3,  'gene':'KDM5A',  'chrom':'chr12', 'pos':394677,    'ref':'G',  'alt':'C',  'variant_type':'missense_variant',       'impact':'MODERATE','cadd_phred':13.88, 'gerp_rs':4.56,  'sift_score':0.06, 'polyphen_score':0.02, 'spliceai_score':0.01},
    {'#':4,  'gene':'C12orf40','chrom':'chr12','pos':40044027,  'ref':'AT', 'alt':'A',  'variant_type':'splice_region_variant',  'impact':'MODERATE','cadd_phred':3.54,  'gerp_rs':-0.78, 'sift_score':None, 'polyphen_score':None,  'spliceai_score':0.04},
    {'#':5,  'gene':'PAPOLA', 'chrom':'chr14', 'pos':97022168,  'ref':'AT', 'alt':'ATT','variant_type':'splice_acceptor_variant','impact':'HIGH',    'cadd_phred':2.71,  'gerp_rs':-0.58, 'sift_score':None, 'polyphen_score':None,  'spliceai_score':0.01},
    {'#':6,  'gene':'RTF1',   'chrom':'chr15', 'pos':41709326,  'ref':'C',  'alt':'A',  'variant_type':'missense_variant',       'impact':'MODERATE','cadd_phred':22.0,  'gerp_rs':3.63,  'sift_score':0.37, 'polyphen_score':0.01, 'spliceai_score':0.01},
    {'#':7,  'gene':'USP7',   'chrom':'chr16', 'pos':8988686,   'ref':'C',  'alt':'G',  'variant_type':'missense_variant',       'impact':'MODERATE','cadd_phred':24.7,  'gerp_rs':3.50,  'sift_score':0.03, 'polyphen_score':0.01, 'spliceai_score':0.01},
    {'#':8,  'gene':'EPG5',   'chrom':'chr18', 'pos':43469840,  'ref':'C',  'alt':'G',  'variant_type':'missense_variant',       'impact':'MODERATE','cadd_phred':21.1,  'gerp_rs':3.30,  'sift_score':0.00, 'polyphen_score':0.55, 'spliceai_score':0.01},
    {'#':9,  'gene':'PIGF',   'chrom':'chr2',  'pos':46839487,  'ref':'GA', 'alt':'G',  'variant_type':'splice_region_variant',  'impact':'MODERATE','cadd_phred':0.43,  'gerp_rs':2.94,  'sift_score':None, 'polyphen_score':None,  'spliceai_score':0.01},
    {'#':10, 'gene':'SPICE1', 'chrom':'chr3',  'pos':113218285, 'ref':'C',  'alt':'G',  'variant_type':'splice_donor_variant',   'impact':'HIGH',    'cadd_phred':25.1,  'gerp_rs':5.94,  'sift_score':None, 'polyphen_score':None,  'spliceai_score':0.91},
    {'#':11, 'gene':'SPICE1', 'chrom':'chr3',  'pos':113218285, 'ref':'C',  'alt':'A',  'variant_type':'splice_donor_variant',   'impact':'HIGH',    'cadd_phred':25.8,  'gerp_rs':5.94,  'sift_score':None, 'polyphen_score':None,  'spliceai_score':0.91},
    {'#':12, 'gene':'SLC2A2', 'chrom':'chr3',  'pos':170727846, 'ref':'T',  'alt':'C',  'variant_type':'missense_variant',       'impact':'MODERATE','cadd_phred':0.0,   'gerp_rs':3.08,  'sift_score':1.00, 'polyphen_score':0.02, 'spliceai_score':0.01},
    {'#':13, 'gene':'ZNF141', 'chrom':'chr4',  'pos':367275,    'ref':'A',  'alt':'G',  'variant_type':'missense_variant',       'impact':'MODERATE','cadd_phred':1.66,  'gerp_rs':1.24,  'sift_score':0.69, 'polyphen_score':0.00, 'spliceai_score':0.01},
    {'#':14, 'gene':'TRIML2', 'chrom':'chr4',  'pos':189020278, 'ref':'C',  'alt':'T',  'variant_type':'missense_variant',       'impact':'MODERATE','cadd_phred':9.62,  'gerp_rs':3.00,  'sift_score':0.01, 'polyphen_score':0.28, 'spliceai_score':0.01},
    {'#':15, 'gene':'TRIML2', 'chrom':'chr4',  'pos':189020278, 'ref':'C',  'alt':'G',  'variant_type':'missense_variant',       'impact':'MODERATE','cadd_phred':19.43, 'gerp_rs':3.00,  'sift_score':0.03, 'polyphen_score':0.02, 'spliceai_score':0.01},
    {'#':16, 'gene':'SREK1',  'chrom':'chr5',  'pos':65473406,  'ref':'G',  'alt':'A',  'variant_type':'missense_variant',       'impact':'MODERATE','cadd_phred':23.7,  'gerp_rs':5.45,  'sift_score':0.00, 'polyphen_score':0.58, 'spliceai_score':0.01},
    {'#':17, 'gene':'SREK1',  'chrom':'chr5',  'pos':65473406,  'ref':'G',  'alt':'C',  'variant_type':'missense_variant',       'impact':'MODERATE','cadd_phred':23.0,  'gerp_rs':5.45,  'sift_score':0.03, 'polyphen_score':0.02, 'spliceai_score':0.01},
    {'#':18, 'gene':'RAD50',  'chrom':'chr5',  'pos':131927550, 'ref':'AT', 'alt':'ATT','variant_type':'splice_region_variant',  'impact':'MODERATE','cadd_phred':1.07,  'gerp_rs':-3.36, 'sift_score':None, 'polyphen_score':None,  'spliceai_score':0.01},
    {'#':19, 'gene':'RARS2',  'chrom':'chr6',  'pos':88224021,  'ref':'C',  'alt':'A',  'variant_type':'missense_variant',       'impact':'MODERATE','cadd_phred':1.45,  'gerp_rs':-3.08, 'sift_score':0.03, 'polyphen_score':0.02, 'spliceai_score':0.01},
    {'#':20, 'gene':'TNRC18', 'chrom':'chr7',  'pos':5385191,   'ref':'A',  'alt':'C',  'variant_type':'splice_donor_variant',   'impact':'HIGH',    'cadd_phred':22.2,  'gerp_rs':5.19,  'sift_score':None, 'polyphen_score':None,  'spliceai_score':1.00},
    {'#':21, 'gene':'DBF4',   'chrom':'chr7',  'pos':87516629,  'ref':'CT', 'alt':'C',  'variant_type':'splice_region_variant',  'impact':'MODERATE','cadd_phred':0.57,  'gerp_rs':1.38,  'sift_score':None, 'polyphen_score':None,  'spliceai_score':0.02},
    {'#':22, 'gene':'LRRCC1', 'chrom':'chr8',  'pos':86022336,  'ref':'CT', 'alt':'C',  'variant_type':'splice_region_variant',  'impact':'MODERATE','cadd_phred':5.45,  'gerp_rs':0.77,  'sift_score':None, 'polyphen_score':None,  'spliceai_score':0.01},
])

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 300)
pd.set_option('display.float_format', '{:.3f}'.format)
print('ALL 22 PHASE 1 VARIANTS — COMPLETE SCORE TABLE')
print('='*90)
display(variants_22)
print()
print('Score availability:')
for col in ['cadd_phred','gerp_rs','sift_score','polyphen_score','spliceai_score']:
    nn = variants_22[col].notna().sum()
    note = '(all variant types)' if col in ['cadd_phred','gerp_rs','spliceai_score'] else '(missense only)'
    print(f'  {col:<22}: {nn}/22  {note}')

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns, numpy as np

score_cols  = ['cadd_phred','gerp_rs','sift_score','polyphen_score','spliceai_score']
col_labels  = ['CADD\nPhred','GERP\nRS','SIFT\nScore','PolyPhen2\nScore','SpliceAI\nScore']
row_labels  = [f"{r['#']}. {r['gene']} ({r['impact'][0]})" for _,r in variants_22.iterrows()]

heat = variants_22[score_cols].copy().astype(float)
heat['sift_score'] = 1 - heat['sift_score'].fillna(0.5)
heat = heat.fillna(0)
for col in heat.columns:
    mn,mx = heat[col].min(), heat[col].max()
    if mx>mn: heat[col] = (heat[col]-mn)/(mx-mn)

annot = variants_22[score_cols].fillna(0).round(2).values

fig, ax = plt.subplots(figsize=(10,10))
sns.heatmap(heat.values, xticklabels=col_labels, yticklabels=row_labels,
    cmap='YlOrRd', vmin=0, vmax=1, linewidths=0.4, linecolor='white',
    cbar_kws={'label':'Normalised Score (0-1, higher=more damaging)'},
    annot=annot, fmt='.2f', annot_kws={'size':8}, ax=ax)
for j,(_,row) in enumerate(variants_22.iterrows()):
    if row['impact']=='HIGH':
        ax.add_patch(plt.Rectangle((0,j),5,1,fill=False,edgecolor='#C0392B',lw=2.5))
ax.set_title('Feature Score Heatmap — 22 CRISPR Off-Target Variants\n(Normalised 0-1 | RED border = HIGH impact)', fontsize=12, fontweight='bold', pad=15)
ax.set_xlabel('Score / Feature', fontsize=11)
ax.set_ylabel('Variant', fontsize=11)
plt.tight_layout()
plt.savefig('figures_scores/score_heatmap_22_variants.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Heatmap saved')

In [ ]:
print('SCORE AVAILABILITY SUMMARY — TRAINING vs TEST vs 22 VARIANTS')
print('='*75)
print(f'{"Score":<22} {"Training (80K)":>16} {"Test (1M)":>11} {"22 Variants":>13} {"ML Feature":>11}')
print('-'*75)
for col in ['cadd_phred','gerp_rs','sift_score','polyphen_score','spliceai_score']:
    tr_pct = train[col].notna().mean()*100 if col in train.columns else 0
    te_pct = test[col].notna().mean()*100  if col in test.columns  else 0
    v22_n  = variants_22[col].notna().sum()
    ml     = '✅ YES' if col=='cadd_phred' else '— annotation'
    print(f'  {col:<20} {tr_pct:>14.1f}% {te_pct:>10.1f}% {v22_n:>9}/22   {ml}')
print()
print('ML model trained on cadd_phred only (single feature, AUC~0.886)')
print('All other scores are supplementary — available for independent use')

In [ ]:
from google.colab import files
import os

out = 'variants_and_scores_22_candidates.csv'
variants_22.to_csv(out, index=False)
print(f'✅ {out}  ({len(variants_22)} variants)')
files.download(out)

for f in sorted(os.listdir('figures_scores')):
    files.download(f'figures_scores/{f}')
    print(f'📥 {f}')